# Edge Deployment Demo

PyTorch → ONNX → INT8 Quantization → Inference Benchmark

Run all 4 cells in order. No GPU needed.

## Cell 1: Clone + Install

In [ ]:
!cd /content && rm -rf edge-deployment-demo && git clone https://github.com/Aeijou37/edge-deployment-demo.git
%cd /content/edge-deployment-demo
!pip install onnxscript onnx onnxruntime gradio -q

## Cell 2: Export ONNX + INT8 Quantize

In [ ]:
import sys
sys.path.insert(0, '.')
from src.export_onnx import create_pointnet_classifier, export_to_onnx, verify_onnx
from src.quantize import quantize_onnx_int8, compare_models

model = create_pointnet_classifier(num_classes=40, num_points=1024)
print('Model created')

fp32_path = export_to_onnx(model, 'models/pointnet_fp32.onnx', num_points=1024)
verify_onnx(model, fp32_path, num_points=1024)
int8_path = quantize_onnx_int8(fp32_path, 'models/pointnet_int8.onnx')
comparison = compare_models(fp32_path, int8_path, num_points=1024, num_samples=50)

## Cell 3: Benchmark

In [ ]:
from src.benchmark import benchmark_all, visualize_benchmark
from IPython.display import Image, display

results = benchmark_all(
    model, 'models/pointnet_fp32.onnx', 'models/pointnet_int8.onnx',
    num_points=1024, num_warmup=10, num_runs=50,
)
img_path = visualize_benchmark(results)
display(Image(img_path))

## Cell 4: Gradio Demo

In [ ]:
from src.app import EdgeDeployApp

app = EdgeDeployApp()
demo = app.build()
demo.launch(share=True)